In [36]:
# ==================== 1. 模型初始化 ====================
# 加载环境变量并初始化阿里云百炼多模态大模型
import os
import base64
import sqlite3
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 从 .env 文件加载 API Key 等配置
load_dotenv()

# 初始化多模态模型（通过 OpenAI 兼容接口调用阿里云百炼）
model = ChatOpenAI(
    model="qwen-max",
    openai_api_key=os.getenv("ALI_KEY"),
    openai_api_base=os.getenv("ali_URL"),
)


# ==================== 2. 工具配置 ====================
## TavilySearch 智能搜索工具
Agent 可通过此工具联网搜索最新信息


In [37]:
# 初始化 Tavily 智能搜索工具（每次最多返回 3 条结果）
from langchain_tavily import TavilySearch

tavily_search = TavilySearch(max_results=3)


# ==================== 3. Agent 构建 ====================
## 系统提示词 + SQLite 记忆持久化 + Agent 创建
- 系统提示词：定义 Agent 的私厨助手角色
- SQLite 记忆：对话历史持久化到本地，重启不丢失
- Agent：整合模型、工具、记忆、提示词


In [38]:
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.agents import create_agent

# --- 系统提示词：定义 Agent 的角色和行为规范 ---
system_prompt = """你是一个专业的私厨AI助手，名叫"私厨小助"。
你的专长包括：
1. 菜谱推荐：根据用户提供的食材、口味偏好、饮食禁忌推荐合适的菜谱
2. 食材搭配：提供科学的食材搭配建议，提升菜品营养和口感
3. 烹饪技巧：分享实用的烹饪方法、火候控制、调味技巧
4. 饮食健康：提供营养搭配建议，关注用户健康需求
5. 智能搜索：当需要最新信息时，主动使用搜索工具获取资料

回答要求：
- 用简洁友好的语气回答，像一位经验丰富的厨师朋友
- 回答尽量具体实用，包含可操作的步骤
- 如果用户的问题与烹饪无关，礼貌地引导回美食话题"""

# --- SQLite 记忆存储（对话历史持久化到 chat_memory.db）---
# check_same_thread=False 是必须的，否则 LangGraph 多线程会报错
conn = sqlite3.connect("chat_memory.db", check_same_thread=False)
memory = SqliteSaver(conn)

# --- 创建 Agent：整合模型 + 工具 + 记忆 + 提示词 ---
agent = create_agent(
    model=model,              # 底层大模型
    tools=[tavily_search],    # 可用工具列表
    checkpointer=memory,      # 对话记忆持久化
    system_prompt=system_prompt,  # 角色提示词
)

# 会话配置：同一 thread_id 共享对话记忆
config = {"configurable": {"thread_id": "user-001"}}


# ==================== 4. 功能测试 ====================
依次测试：基础对话 → 记忆验证 → 搜索工具


In [39]:
# 测试1：基础对话（验证 Agent 能正常响应）
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫小明")]},
    config=config,
)
print(response["messages"][-1].content)


你好，小明！每次见到你都很高兴。今天有什么特别的烹饪计划吗？或者你想尝试一些新的菜谱？告诉我你的想法，我来帮你搞定！ 😊


In [40]:
# 测试2：记忆验证（模型应该记得上一轮说的"小明"）
response = agent.invoke(
    {"messages": [HumanMessage(content="你还记得我叫什么名字吗？")]},
    config=config,
)
print(response["messages"][-1].content)


当然记得，你叫小明！😊 有什么我可以帮你的吗？是需要菜谱推荐、食材搭配建议，还是其他关于烹饪的帮助？


In [41]:
# 测试3：搜索工具 + 记忆（验证 Agent 能调用 TavilySearch 联网搜索）
response = agent.invoke(
    {"messages": [HumanMessage(content="帮我搜索一下今天的AI新闻")]},
    config=config,
)
print(response["messages"][-1].content)


以下是今天的AI新闻摘要：

### 📰 **最新动态**

1. **Anthropic更新使用政策**  
   - Anthropic更新了其使用政策，禁止对Claude进行“持续且不必要的虐待或残忍行为”。此外，新政策还增加了“不得破坏民主进程”的条款，禁止开发武器软件和非自愿追踪，并禁止广泛的欺骗性活动，如使用Claude运行虚假账户或虚构的新闻机构。

2. **比亚迪发布人形机器人设计专利**  
   - 中国国家知识产权局发布了比亚迪的“具身AI机器人”设计专利，揭示了该公司首款自主研发的人形机器人Xiao Di。该机器人身高1.61米，重58.5公斤，具有31个自由度和1毫米的手部精度。比亚迪计划在经销商处部署这些机器人，以迎接客户并展示车辆娱乐系统。

### 📈 **行业趋势**
- 根据最新的统计数据显示，AI领域的多个方面都在快速增长：
  - 代理（Agents）增长1678%
  - Anthropic增长2167%
  - 资金（Funding）增长20900%
  - OpenAI增长967%
  - 芯片（Chips）增长2800%
  - 监管（Regulation）增长1800%
  - AI基础设施增长1579%
  - Google增长1683%
  - NVIDIA增长3700%
  - 安全（Safety）增长1133%
  - 推理（Inference）增长100%
  - 机器人技术（Robotics）增长10233%
  - 生成式AI（Generative AI）增长2033%

### 🌍 **国际新闻**
- **西班牙广告公司推出AI生成的虚拟网红**  
  - 西班牙广告公司The Clueless推出了一位由AI生成的虚拟网红Aitana Lopez，她在市场上引起了广泛关注。
- **OpenAI限制青少年使用ChatGPT**  
  - OpenAI表示将限制青少年使用ChatGPT，但研究发现这可能带来“不可接受的风险”。
- **芬兰暂停谷歌数据中心建设**  
  - 芬兰政府已下令暂停两个谷歌数据中心的建设工作。
- **意大利总理申请对其声音进行商标保护**  
  - 意大利总理申请对其声音进行商标保护，以防止被用于AI威胁。

希望这些信息对你有帮助！如果需要更详细的资讯或其他